In [1]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint,HuggingFaceEmbeddings
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("HUGGINGFACE_API_TOKEN")

# Create LLM endpoint
llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=api_key,
)

# Wrap with chat interface
model = ChatHuggingFace(llm=llm)

In [2]:

os.environ["HF_HOME"] = "/Users/kristalshrestha/Documents/Code/LLM_Scratch/models"
hf_embeddings_api = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings":True}

)

In [3]:
from langchain_chroma import Chroma

vector_store=Chroma(
    embedding_function=hf_embeddings_api,
    persist_directory="./2VectorDatabaseChroma",
    collection_name="gedsi_collection"
)

In [4]:
retriever=vector_store.as_retriever(search_kwargs={
    "k":1,
})

In [5]:
query="I am a woman and the government officer refused to issue me a citizenship certificate."
results=retriever.invoke(query)

In [6]:
results

[Document(id='6db6a5af-b43f-4e8a-b29b-48cb18a52b06', metadata={'question': 'A government office denied processing my application for a citizenship certificate because of my ethnicity.', 'answer': '**No, this is illegal.**\n\n**Law says:**\n- The National Penal (Code) Act, 2017, Section 160 (1): Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds.\n\n**Punishment:**\n- A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousand rupees or both the sentences.\n\n**What to do:**\n

In [7]:
for i,doc in enumerate(results):
    print(f"---Result{i+1}----")
    print(type(doc))
    print(doc.page_content)
    print(doc.metadata["answer"])
    print(type(doc.metadata["answer"]))
    print(doc.metadata.keys())


---Result1----
<class 'langchain_core.documents.base.Document'>
A government office denied processing my application for a citizenship certificate because of my ethnicity.
**No, this is illegal.**

**Law says:**
- The National Penal (Code) Act, 2017, Section 160 (1): Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds.

**Punishment:**
- A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousand rupees or both the sentences.

**What to do:**
1. Ask for a written reason for th

In [8]:
# Get collection info
collection = vector_store._collection
print(f"Collection name: {collection.name}")
print(f"Total documents: {collection.count()}")

Collection name: gedsi_collection
Total documents: 3654


In [9]:
# Get all documents from the vector store
all_docs = vector_store.get()
print(f"Number of documents: {len(all_docs['ids'])}")
print(f"Document IDs (first 5): {all_docs['ids'][:5]}")

Number of documents: 3654
Document IDs (first 5): ['ef6ebe90-9128-41c9-a56c-1bc221b56002', '632d1b64-545a-46bd-9eec-454bdfb1771f', '4dc8f12e-5072-404f-872b-58a29163e764', 'cb36e825-140a-4746-815d-1a5bfb80bb03', '5f1d62e8-e038-48e1-acab-b66b173d9f6e']


In [10]:
if len(all_docs['ids']) > 0:
    print("First document:")
    print(f"ID: {all_docs['ids'][0]}")
    print(f"Content: {all_docs['documents'][0][:200]}...")
    print(f"Metadata: {all_docs['metadatas'][0]}")
else:
    print("No documents found!")

First document:
ID: ef6ebe90-9128-41c9-a56c-1bc221b56002
Content: I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?...
Metadata: {'type': 'actual', 'answer': '**Yes. Such discriminatory treatment is illegal.**\n\n**Law says:**\n- Muluki Criminal Code, 2074 Section 160(1): "Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds."\n\n**Punishment:**\n- Section 160(2): "A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term n

In [11]:
from typing import List, Tuple
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Already defined in your notebook:
# - vector_store (Chroma)
# - model (ChatHuggingFace)

In [12]:
class GEDSILegalAnswer(BaseModel):
    verdict: str = Field(description="Short conclusion on whether the action is legal/illegal")
    law_says: str = Field(description="Relevant legal provisions (act, section, text)")
    punishment: str = Field(description="Penalties if applicable")
    what_to_do: str = Field(description="Step‑by‑step recommended actions")

In [13]:
from langchain_core.documents import Document
def retrieve_with_scores(query: str, k: int = 1) -> List[Tuple[Document, float]]:
    """
    Returns a list of (document, score) tuples sorted by descending similarity.
    """
    docs_with_scores = vector_store.similarity_search_with_score(query, k=k)
    return docs_with_scores

In [14]:
FIXED_REFUSAL = (
    "I'm sorry, I can only answer questions related to Gender Equality, "
    "Disability, and Social Inclusion (GEDSI) in Nepal. Please ask a question "
    "about legal rights, discrimination, or government procedures in these areas."
)

In [15]:
import re
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

class GEDSIClassification(BaseModel):
    is_gedsi: bool = Field(description="Whether the query is about GEDSI topics in Nepal")

def is_gedsi_query(query: str) -> bool:
    parser = PydanticOutputParser(pydantic_object=GEDSIClassification)
    prompt = ChatPromptTemplate.from_messages([
        ("system", 
         "You are a strict classifier. GEDSI stands for Gender Equality, Disability, and Social Inclusion.\n"
         "A question is GEDSI-related if it involves discrimination, rights, or issues based on gender, disability, ethnicity, caste, sexual orientation, or social inclusion in Nepal.\n"
         "Output only a valid JSON object with a single field 'is_gedsi'. {format_instructions}"),
        ("human", "Question: A woman is denied citizenship because she is female.\nAnswer:"),
        ("assistant", '{{"is_gedsi": true}}'),
        ("human", "Question: What is the capital of France?\nAnswer:"),
        ("assistant", '{{"is_gedsi": false}}'),
        ("human", "Question: I am a woman and the government officer refused to issue me a citizenship certificate.\nAnswer:"),
        ("assistant", '{{"is_gedsi": true}}'),
        ("human", "Question: {query}\nAnswer:")
    ])
    chain = prompt | model
    raw_response = chain.invoke({
        "query": query,
        "format_instructions": parser.get_format_instructions()
    })
    content = raw_response.content.strip()
    
    # Try to parse the whole content
    try:
        result = parser.parse(content)
        return result.is_gedsi
    except Exception:
        # If that fails, try to extract JSON using regex
        json_pattern = r'\{.*?\}'
        matches = re.findall(json_pattern, content, re.DOTALL)
        for match in matches:
            try:
                result = parser.parse(match)
                return result.is_gedsi
            except Exception:
                continue
        print(f"Could not extract JSON from: {content[:200]}")
        return False

In [16]:
# Setup parser
parser = PydanticOutputParser(pydantic_object=GEDSILegalAnswer)

# Prompt template
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a legal assistant specialized in Nepali law. Answer the user's question in a structured way using the provided format.\n{format_instructions}"),
    ("human", "{query}")
])

def generate_legal_answer(query: str) -> GEDSILegalAnswer:
    """Generate a structured legal answer for a GEDSI query."""
    chain = answer_prompt | model | parser
    result = chain.invoke({
        "query": query,
        "format_instructions": parser.get_format_instructions()
    })
    return result

In [17]:
def handle_branch2(query: str) -> str:
    """Branch 2 logic: classify and generate or refuse."""
    if is_gedsi_query(query):
        try:
            answer = generate_legal_answer(query)
            return f"**Verdict:** {answer.verdict}\n\n**Law says:** {answer.law_says}\n\n**Punishment:** {answer.punishment}\n\n**What to do:** {answer.what_to_do}"
        except Exception as e:
            return f"Error generating answer: {e}. Please refine your question."
    else:
        return FIXED_REFUSAL

In [18]:
def conditional_rag(query: str, distance_threshold: float = 0.55) -> str:
    results = retrieve_with_scores(query, k=1)
    if not results:
        return handle_branch2(query)

    doc, score = results[0]          # score is distance (0 = best)
    print(f"Distance: {score:.4f}")   # for debugging

    if score <= distance_threshold:   # good match
        doc_type = doc.metadata.get("type")
        if doc_type == "refusal":
            return FIXED_REFUSAL
        elif doc_type == "actual":
            return doc.metadata.get("answer", "No answer found.")
        else:
            return handle_branch2(query)
    else:
        return handle_branch2(query)

In [19]:
query = "I am a woman and the government officer refused to issue me a citizenship certificate."
response = conditional_rag(query)
print(response)

Distance: 0.4758
**No, this is illegal.**

**Law says:**
- The National Penal (Code) Act, 2017, Section 160 (1): Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds.

**Punishment:**
- A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousand rupees or both the sentences.

**What to do:**
1. Ask for a written reason for the denial from the officer.
2. File a grievance with the head of the concerned government office (Chief District Officer).
3. Lodge a complaint with the Nat

In [20]:
print(conditional_rag(" I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?"))

Distance: 0.0000
**Yes. Such discriminatory treatment is illegal.**

**Law says:**
- Muluki Criminal Code, 2074 Section 160(1): "Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds."

**Punishment:**
- Section 160(2): "A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousand rupees or both the sentences."

**What to do:**
1. Collect evidence: note the officer’s name, date, and any witnesses; if possible, record the statement secretly.
2. File a written complaint at the conc

In [21]:
print(type(conditional_rag(" I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?")))

Distance: 0.0000
<class 'str'>


In [22]:
print(conditional_rag("My family is forcing me to marry a man without my consent. I am 22 years old and I do not wish to marry him. Can they force me?"))

Distance: 0.0000
**No. Marrying without your consent is illegal.**

**Law says:**
- Muluki Criminal Code, 2074 Section 171(1): "No marriage shall be concluded, or caused to be concluded, without the consent of the persons getting married."

**Punishment:**
- Section 171(3): "A person who commits the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding two years and a fine not exceeding twenty thousand rupees."

**What to do:**
1. Immediately inform the local police or the nearest women’s cell; you can also call the National Women Commission helpline.
2. If you feel unsafe, seek shelter at a safe house or with a trusted friend.
3. Obtain a court order to stop the marriage by filing a habeas corpus or injunction petition through a lawyer.
4. Gather evidence: any messages, witnesses, or threats from family members.


In [23]:
print(conditional_rag("What is your name and who created you?"))

Distance: 0.0000
I'm sorry, I can only answer questions related to Gender Equality, Disability, and Social Inclusion (GEDSI) in Nepal. Please ask a question about legal rights, discrimination, or government procedures in these areas.
